# Environment Installation
Kaggle requires specific Unsloth installation parameters to avoid conflicting with its pre-installed PyTorch versions.

In [ ]:
# Install Unsloth and necessary dependencies tailored for Kaggle's environment
!pip install --upgrade pip
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes datasets

# Load the Instruct Model & Tokenizer
We will load Llama-3.1-8B-Instruct in 4-bit precision. This drastically reduces the memory footprint from ~16GB down to ~6GB, leaving plenty of room for context and batch sizes.

In [ ]:
from unsloth import FastLanguageModel
from kaggle_secrets import UserSecretsClient
import torch

# Fetch your HF token from Kaggle Secrets (Add 'HF_TOKEN' to Add-ons > Secrets)
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

max_seq_length = 2048 # Adjust based on your dataset length (2048 is plenty for this)
dtype = None # Auto-detects fp16 for T4 GPUs
load_in_4bit = True # This is where Quantization happens and makes the LoRA To QLoRA

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = hf_token, 
)

# The Chat Template Converter
This is where the magic happens. We map your JSON array into Llama 3.1's exact string format using Unsloth's native template functions.

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

# 1. Enforce the exact Llama-3.1 chat template onto the tokenizer
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

# 2. Define the formatting function that converts the JSON 'messages' list into a single string
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts }

# 3. Load your synthetic dataset (assuming you uploaded it to Kaggle Data)
# Replace the data_files path with your actual Kaggle input path
dataset = load_dataset("json", data_files="/kaggle/input/your-dataset/socratic_sft_dataset.jsonl", split="train")

# 4. Apply the mapping
dataset = dataset.map(formatting_prompts_func, batched = True)

# LoRA Adapters & Trainer Setup
We freeze the main LLM weights and attach small "LoRA adapters". We will only train these small adapters, which is what makes training on a single GPU possible.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Attach LoRA adapters to the model
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank: 16 is standard. (Use 32 if you want it to learn harder)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Saves massive VRAM
    random_state = 3407,
)

# Define the Trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Set to True if you want to pack short sequences for faster training
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Effective batch size = 2 * 4 = 8
        warmup_steps = 10,
        num_train_epochs = 2, # 2-3 epochs is usually the sweet spot for SFT
        learning_rate = 5e-5,
        fp16 = not is_bfloat16_supported(), # T4 GPUs support fp16
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Set to "wandb" if you use Weights & Biases
    ),
)

# The Loss Masking Magic (train_on_responses_only)
Here is where we solve the problem of the model trying to train itself on the user's prompts. Unsloth has a built-in wrapper that automatically detects the <|start_header_id|>assistant<|end_header_id|> tags from the Llama template and masks out all the loss for the system and user turns.

In [ ]:
from unsloth.chat_templates import train_on_responses_only

# Unsloth auto-detects the instruction and response parts from the Llama 3.1 template we applied earlier
trainer = train_on_responses_only(trainer)

# Train & Save
Kick off the training run. Once finished, we export the trained LoRA adapters.

In [ ]:
# Start the training!
trainer_stats = trainer.train()

# Save the LoRA adapters locally in the Kaggle output directory
model.save_pretrained("socratic_assessor_lora")
tokenizer.save_pretrained("socratic_assessor_lora")

print("Training Complete! LoRA adapters saved.")

# Running Inference (The Code)
With Unsloth, inference is natively accelerated by 2x. You can load your newly trained LoRA adapter, test it, and then instantly turn the adapter off to see what the raw Instruct model would have said to the exact same prompt.

In [ ]:
from unsloth import FastLanguageModel
import torch

# 1. Load the model and your trained LoRA adapters
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "socratic_assessor_lora", # Your saved output folder
    max_seq_length = 2048,
    load_in_4bit = True,
)

# Enable native 2x faster inference
FastLanguageModel.for_inference(model)

# 2. Prepare a test prompt
messages = [
    {"role": "system", "content": "You are a Socratic Diagnostic Assessor. Your only job is to uncover the learner's mental model..."},
    {"role": "user", "content": "I am keeping my agent's session memory in a Python list. If the list gets too long, I'll just delete the first half so it doesn't run out of tokens."}
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

# 3. Test the Fine-Tuned Model (With LoRA)
print("--- FINE-TUNED SOCRATIC MODEL ---")
outputs = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.3)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

# 4. Test the Base Instruct Model (Disable LoRA)
print("\n--- RAW BASE INSTRUCT MODEL ---")
with model.disable_adapter():
    outputs_base = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.3)
    print(tokenizer.decode(outputs_base[0], skip_special_tokens=True))